# RL Final Project

In [66]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import matplotlib.pyplot as plt

class MazeEnv(gym.Env):
    def __init__(self):
        super(MazeEnv, self).__init__()
        self.maze = np.array([
            [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
            [1, 'S', 1, 0, 0, 0, 1, 0, 0, 1],
            [1, 0, 1, 0, 1, 0, 1, 0, 1, 1],
            [1, 0, 0, 0, 1, 0, 0, 0, 0, 1],
            [1, 1, 1, 0, 1, 1, 1, 1, 0, 1],
            [1, 0, 1, 'G', 0, 0, 0, 1, 0, 1],
            [1, 0, 0, 0, 1, 1, 0, 0, 0, 1],
            [1, 1, 1, 0, 1, 0, 1, 1, 0, 1],
            [1, 0, 0, 0, 0, 0, 1, 'E', 0, 1],
            [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
        ])

        self.start_pos = [1, 1]
        self.goal_pos = [8, 7]
        self.sub_goal_pos = [5, 3]
        self.agent_pos = self.start_pos

        self.action_space = spaces.Discrete(4)  # 4 possible actions: up, down, left, right

        # Flatten the 10x10 maze into a 1D array of 100 states (0 to 99)
        self.num_states = self.maze.shape[0] * self.maze.shape[1]
        self.observation_space = spaces.Discrete(self.num_states)

        self.reset()

    def reset(self):
        self.agent_pos = self.start_pos
        return self._get_obs()

    def step(self, action):
        next_pos = list(self.agent_pos)

        # Up, Down, Left, Right
        if action == 0:  # Up
            next_pos[0] -= 1
        elif action == 1:  # Down
            next_pos[0] += 1
        elif action == 2:  # Left
            next_pos[1] -= 1
        elif action == 3:  # Right
            next_pos[1] += 1

        if self._is_valid_move(next_pos):
            self.agent_pos = next_pos

        done = False
        reward = -1  # Default penalty for each move

        if self.agent_pos == self.sub_goal_pos:
            reward = 10  # Reward for reaching sub-goal
        if self.agent_pos == self.goal_pos:
            reward = 100  # Reward for reaching final goal
            done = True

        return self._get_obs(), reward, done

    def _get_obs(self):
        """Returns the current state as a unique integer for the Q-table."""
        # Convert (row, col) position to a single integer
        state = self._pos_to_state(self.agent_pos)
        return state

    def _is_valid_move(self, pos):
        """Checks if the move is within bounds and not a wall."""
        if 0 <= pos[0] < self.maze.shape[0] and 0 <= pos[1] < self.maze.shape[1]:
            return self.maze[tuple(pos)] != 1
        return False

    def _pos_to_state(self, pos):
        """Converts a (row, col) position to a unique state index."""
        return pos[0] * self.maze.shape[1] + pos[1]

    def _state_to_pos(self, state):
        """Converts a state index back to a (row, col) position."""
        row = state // self.maze.shape[1]
        col = state % self.maze.shape[1]
        return [row, col]

    def render(self):
        """Renders the maze with the agent, start, sub-goal, and end goal."""
        maze_to_render = np.copy(self.maze)

        # Replace markers for rendering (we'll use text annotations for 'S', 'G', 'E')
        maze_to_render[maze_to_render == 'S'] = 0  # Replace 'S' with 0 (empty space)
        maze_to_render[maze_to_render == 'G'] = 0  # Replace 'G' with 0
        maze_to_render[maze_to_render == 'E'] = 0  # Replace 'E' with 0
        maze_to_render = maze_to_render.astype(int)  # Convert to integer array

        # Mark agent's position with a different integer (e.g., 2)
        maze_to_render[tuple(self.agent_pos)] = 2  # Agent's position (marked with 2)

        # Render using imshow
        plt.imshow(maze_to_render, cmap="cool", origin="upper")
        
        # Add text labels for start, sub-goal, and end goal
        plt.text(1, 1, 'S', ha='center', va='center', color='black', fontsize=12)  # Start
        plt.text(5, 3, 'G', ha='center', va='center', color='black', fontsize=12)  # Sub-goal
        plt.text(8, 7, 'E', ha='center', va='center', color='black', fontsize=12)  # End
        
        plt.title("Maze Environment")
        plt.show()


## Get the number of states and actions

In [67]:
env = MazeEnv()

num_states = 100
num_actions = env.action_space.n

## Initialize the Q-table 

In [57]:
Q = np.zeros((num_states, num_actions))

In [58]:
reward_per_random_episode = []

## How to update the Q-table

In [68]:
def update_q_table(state, action, reward,next_state):

    old_value = Q[state,action]
    next_value = max(Q[next_state])

    new_value = (1-alpha)* old_value + alpha*(reward + (gamma*next_value))

    Q[state,action] = new_value

alpha = 0.1
gamma = 0.8

## Interaction loop

In [73]:
num_episodes = 10000
Q = np.zeros((num_states, num_actions))
for i in range(num_episodes):

    state = env.reset()
    terminated = False
    episode_reward = 0

    while not terminated:

        action = env.action_space.sample()

        next_state, reward, terminated = env.step(action)
        
        update_q_table(state,action,reward,next_state)

        episode_reward += reward

        state = next_state

    reward_per_random_episode.append(episode_reward)

    

In [74]:
Q

array([[  0.53985283,   1.92481604,   0.53985283,   1.92481604],
       [  1.92481604,   3.65602005,   0.53985283,   3.65602005],
       [  3.65602005,   5.82002506,   1.92481604,   5.82002506],
       [  5.82002506,   8.52503132,   3.65602005,   4.01943132],
       [  4.01943132,   6.27428915,   5.82002506,   6.27428915],
       [  6.27428915,   9.09286144,   4.01943132,   9.09286144],
       [  9.09286144,  12.6160768 ,   6.27428915,  12.6160768 ],
       [ 12.6160768 ,  17.020096  ,   9.09286144,   9.09286144],
       [  9.09286144,  12.6160768 ,  12.6160768 ,   6.27428915],
       [  6.27428915,   9.09286144,   9.09286144,   6.27428915],
       [  0.53985283,   3.65602005,   1.92481604,   3.65602005],
       [  1.92481604,   5.82002506,   1.92481604,   5.82002506],
       [  3.65602005,   8.52503132,   3.65602005,   8.52503132],
       [  5.82002506,  11.90628915,   5.82002506,   6.27428915],
       [  4.01943132,   9.09286144,   8.52503132,   9.09286144],
       [  6.27428915,  12